In [1]:
# ================================================================================================
# Revenue Regression Pipeline — Predicting Order Value
# ================================================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble        import RandomForestRegressor
from sklearn.linear_model    import LinearRegression
from sklearn.preprocessing   import OneHotEncoder, RobustScaler
from sklearn.impute          import SimpleImputer
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score
from scipy.sparse            import hstack
import xgboost as xgb

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# ================================================================================================
# 1. Load and merge data
# ================================================================================================
train = pd.read_csv('../data/raw/train.csv', sep='|')
items = pd.read_csv('../data/raw/items.csv', sep='|')
df = train.merge(items, on='pid', how='left', validate='m:1')
print(f"Full dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

Full dataset: 2,756,003 rows x 21 columns


In [3]:
# ================================================================================================
# 2. Filter to orders only — regression only makes sense where a purchase occurred
# ================================================================================================
df_orders = df[df['order'] == 1].copy()
df_orders = df_orders.sort_values(['day', 'lineID']).reset_index(drop=True)
print(f"Orders only: {df_orders.shape[0]:,} rows")
print(f"Revenue range: {df_orders['revenue'].min():.2f} — {df_orders['revenue'].max():.2f}")
print(f"Revenue mean: {df_orders['revenue'].mean():.2f}")
print(f"Revenue median: {df_orders['revenue'].median():.2f}")

Orders only: 705,090 rows
Revenue range: 0.07 — 887.70
Revenue mean: 14.66
Revenue median: 9.86


In [4]:
# ================================================================================================
# 3. Feature Engineering
# ================================================================================================

# Price features
df_orders['priceRatio'] = (
    df_orders['price'] / df_orders['rrp'].replace(0, np.nan)
).fillna(1.0)

df_orders['priceVsCompetitor'] = (
    df_orders['price'] / df_orders['competitorPrice'].replace(0, np.nan)
).fillna(1.0)

df_orders['priceDiscount'] = (df_orders['rrp'] - df_orders['price']).clip(lower=0)
df_orders['missingCompetitorPrice'] = df_orders['competitorPrice'].isnull().astype(int)

# Cyclical weekday encoding
df_orders['weekDay_raw'] = (df_orders['day'] % 7).replace({0: 7})
df_orders['weekDay_sin'] = np.sin(2 * np.pi * df_orders['weekDay_raw'] / 7)
df_orders['weekDay_cos'] = np.cos(2 * np.pi * df_orders['weekDay_raw'] / 7)
df_orders.drop(columns=['weekDay_raw'], inplace=True)

# Product age
df_orders['first_seen'] = df_orders.groupby('pid')['day'].transform('min')
df_orders['product_age_days'] = df_orders['day'] - df_orders['first_seen']
df_orders.drop(columns=['first_seen'], inplace=True)

# Interaction features
df_orders['adFlag_x_priceRatio']  = df_orders['adFlag'] * df_orders['priceRatio']
df_orders['regulated_generic']    = df_orders['salesIndex'] * df_orders['genericProduct']

print("Feature engineering complete.")
print(f"Columns: {list(df_orders.columns)}")

Feature engineering complete.
Columns: ['lineID', 'day', 'pid', 'adFlag', 'availability', 'competitorPrice', 'click', 'basket', 'order', 'price', 'revenue', 'manufacturer', 'group', 'content', 'unit', 'pharmForm', 'genericProduct', 'salesIndex', 'category', 'campaignIndex', 'rrp', 'priceRatio', 'priceVsCompetitor', 'priceDiscount', 'missingCompetitorPrice', 'weekDay_sin', 'weekDay_cos', 'product_age_days', 'adFlag_x_priceRatio', 'regulated_generic']


In [5]:
# ================================================================================================
# 4. Define target and feature sets
# ================================================================================================
TARGET = 'revenue'

NUM_FEATURES = [
    'price', 'rrp', 'competitorPrice',
    'priceRatio', 'priceVsCompetitor', 'priceDiscount',
    'missingCompetitorPrice', 'weekDay_sin', 'weekDay_cos',
    'adFlag_x_priceRatio', 'regulated_generic', 'product_age_days'
]

CAT_FEATURES = [
    'adFlag', 'availability', 'content', 'unit',
    'pharmForm', 'genericProduct', 'salesIndex'
]

DROP_COLS = [
    'lineID', 'day', 'pid', 'order',
    'click', 'basket', 'revenue',
    'campainIndex', 'manufacturer', 'group', 'category'
]

print(f"Numeric features: {len(NUM_FEATURES)}")
print(f"Categorical features: {len(CAT_FEATURES)}")

Numeric features: 12
Categorical features: 7


In [6]:
# ================================================================================================
# 5. Train / Test Split — time-based 80/20
# ================================================================================================
y = df_orders[TARGET]
X = df_orders.drop(columns=[c for c in DROP_COLS if c in df_orders.columns])

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Train revenue mean: {y_train.mean():.2f}  |  Test revenue mean: {y_test.mean():.2f}")

Train size: 564,072  |  Test size: 141,018
Train revenue mean: 14.58  |  Test revenue mean: 14.99


In [7]:
# ================================================================================================
# 6. Preprocessing — impute, scale, encode
# ================================================================================================

# Filter feature lists to columns that exist
active_num = [f for f in NUM_FEATURES if f in X_train.columns]
active_cat = [f for f in CAT_FEATURES if f in X_train.columns]

# Fill missing categoricals
fill = {col: 'Missing' for col in active_cat}
for d in [X_train, X_test]:
    d.fillna(fill, inplace=True)
    d[active_cat] = d[active_cat].astype(str)

# Impute and scale numericals
imputer = SimpleImputer(strategy='median')
scaler  = RobustScaler()

X_train_num = scaler.fit_transform(imputer.fit_transform(X_train[active_num]))
X_test_num  = scaler.transform(imputer.transform(X_test[active_num]))

# Encode categoricals
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_cat = encoder.fit_transform(X_train[active_cat])
X_test_cat  = encoder.transform(X_test[active_cat])

# Combine
X_train_proc = hstack([X_train_num, X_train_cat]).tocsr()
X_test_proc  = hstack([X_test_num,  X_test_cat]).tocsr()

print(f"Preprocessed train shape: {X_train_proc.shape}")
print(f"Preprocessed test shape:  {X_test_proc.shape}")

Preprocessed train shape: (564072, 791)
Preprocessed test shape:  (141018, 791)


In [ ]:
# ================================================================================================
# 7. Train and evaluate models
# ================================================================================================

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    rmse  = np.sqrt(mean_squared_error(y_te, preds))
    mae   = mean_absolute_error(y_te, preds)
    r2    = r2_score(y_te, preds)
    print(f"\n{name}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  R²   : {r2:.4f}")
    return model, preds, {'model': name, 'rmse': rmse, 'mae': mae, 'r2': r2}

models = {
    'Linear Regression':       LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost Regressor':       xgb.XGBRegressor(n_estimators=200, learning_rate=0.05,
                                                  max_depth=6, random_state=42, n_jobs=-1),
}

results = []
trained_models = {}
all_preds = {}

print("=== Regression Results on Test Set ===")
for name, model in models.items():
    trained_model, preds, metrics = evaluate_model(
        name, model, X_train_proc, y_train, X_test_proc, y_test
    )
    trained_models[name] = trained_model
    all_preds[name] = preds
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values('rmse')
print("\n=== Summary Table ===")
print(results_df.to_string(index=False))
results_df.to_csv('../reports/regression_results.csv', index=False)

=== Regression Results on Test Set ===

Linear Regression
  RMSE : 9.8645
  MAE  : 4.3483
  R²   : 0.6321


In [ ]:
# ================================================================================================
# 8. Visualisations
# ================================================================================================

# 8a. Actual vs Predicted — best model (lowest RMSE)
best_model_name = results_df.iloc[0]['model']
best_preds = all_preds[best_model_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted
cap = np.percentile(y_test, 99)
mask = y_test <= cap
axes[0].scatter(y_test[mask], best_preds[mask], alpha=0.3, s=8, color='#4C72B0')
axes[0].plot([0, cap], [0, cap], 'r--', linewidth=1.5, label='perfect prediction')
axes[0].set_xlabel('Actual Revenue')
axes[0].set_ylabel('Predicted Revenue')
axes[0].set_title(f'Actual vs Predicted Revenue\n({best_model_name})')
axes[0].legend()

# Residuals distribution
residuals = y_test.values - best_preds
axes[1].hist(residuals, bins=60, color='#DD8452', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residual Distribution\n({best_model_name})')

plt.suptitle('Regression Diagnostics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/regression_diagnostics.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 8b. Feature importance — Random Forest (tree-based, sparse-compatible)
rf_model = trained_models['Random Forest Regressor']
num_names = active_num
cat_names = list(encoder.get_feature_names_out(active_cat))
all_feature_names = num_names + cat_names

importances = pd.Series(rf_model.feature_importances_, index=all_feature_names)
top20 = importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top20.index[::-1], top20.values[::-1], color='#4C72B0')
ax.set_title('Top 20 Feature Importances — Random Forest Regressor')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../reports/regression_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 8c. Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics_to_plot = ['rmse', 'mae', 'r2']
titles = ['RMSE (lower is better)', 'MAE (lower is better)', 'R² (higher is better)']
colors = ['#4C72B0', '#DD8452', '#55A868']

for ax, metric, title, color in zip(axes, metrics_to_plot, titles, colors):
    ax.barh(results_df['model'], results_df[metric], color=color, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel(metric.upper())

plt.suptitle('Regression Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/regression_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nRegression pipeline completed. Outputs saved to ../reports/")